
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>



# Data Skipping
In this demo, we are going to work with Liquid Clustering, a Delta Lake optimization feature that replaces table partitioning and ZORDER to simplify data layout decisions and optimize query performance. It provides flexibility to redefine clustering keys without rewriting data. Refer to the [documentation](https://docs.databricks.com/en/delta/clustering.html) for more information.

**PLEASE NOTE:** this demo relies on a specific set of tables. If this course is being led by an instructor, these tables will have been already set up to save time. The datasets that demonstrate Liquid Clustering are large, and the clusters we use with demos and labs are small.

If you are taking this course as a self-paced course on your own, you will need to run the [Flight Data Generation notebook]($./Includes/Flight Data Generation) first.

## Getting Started
Run the next cell to set up the lesson.

In [0]:
%run ./Includes/Classroom-Setup-04.2

Run the following cell, which will set a Spark configuration variable that disables caching. Turning caching off makes the effect of the optimizations more apparent.

In [0]:
spark.conf.set('spark.databricks.io.cache.enabled', False)

Now let's see a count of the number of records in the `flights` table.

In [0]:
%sql
SELECT COUNT(*) FROM dbacademy.flights

## Liquid Clustering
Delta Lake liquid clustering replaces table partitioning and ZORDER to simplify data layout decisions and optimize query performance. Liquid clustering provides flexibility to redefine clustering keys without rewriting existing data, allowing data layout to evolve alongside analytic needs over time.  
  
Databricks recommends liquid clustering for all new Delta tables. The following are examples of scenarios that benefit from clustering:
- Tables often filtered by high cardinality columns.
- Tables with significant skew in data distribution.
- Tables that grow quickly and require maintenance and tuning effort.
- Tables with concurrent write requirements.
- Tables with access patterns that change over time.
- Tables where a typical partition key could leave the table with too many or too few partitions.

## About the Following Tables

We will be querying three tables: 
* `flights`, which does not use liquid clustering
* `flights_cluster_id`, which has been clustered by `id`
* `flights_cluster_id_flight_num`, which has been clustered by `id` and `FlightNum`. 

Before proceeding, let's remind ourselves what the data looks like.

In [0]:
%sql
SELECT * FROM dbacademy.flights

## Unclustered Table
Run the following three cells and note the time it takes to run each cell.  
  
These queries are already quite fast without using clustering, considering we are using a small cluster, and the tables are 9 GB of data. But there is room for improvement.

In [0]:
%sql
SELECT AVG(ArrDelay) FROM dbacademy.flights WHERE UniqueCarrier = 'TW'

In [0]:
%sql
SELECT AVG(ArrDelay) FROM dbacademy.flights WHERE FlightNum = 1890

In [0]:
%sql
SELECT * FROM dbacademy.flights WHERE id = 1125281431554

For all three queries above, drop open the triangle next to **Spark Jobs** and click **View** next to the first job. Drop open **Completed Stages** and note the amount of data that had to be pulled from the data store (in the **Input** column).

## Clustered by ID
Run the following three queries. These queries pull data from the `flights_cluster_id` table. This table is exactly the same as the one we used in the queries above, except that we have enabled liquid clustering by adding `CLUSTER BY (id)` when the table was created.  
  
Note the following:
- When we query by the clustered column (id), we see a significant improvement in query performance
- We don't see a degredation in performance on queries against unclustered columns  

View the first job in the query that filters by `id` and note the amount of data that was skipped. You can also see in the query plan that the filter was pushed down.

In [0]:
%sql
SELECT AVG(ArrDelay) FROM dbacademy.flights_cluster_id WHERE UniqueCarrier = 'TW'

In [0]:
%sql
SELECT AVG(ArrDelay) FROM dbacademy.flights_cluster_id WHERE FlightNum = 1890

In [0]:
%sql
SELECT * FROM dbacademy.flights_cluster_id WHERE id = 1125281431554

## Clustered by ID and FlightNum

Run the following three queries. These queries pull data from the `flights_cluster_id_flightnum` table. This table is clustered by both the `id` and `flight_num` columns.  

Note the following:
- We still don't have any degredation on unclustered columns. Had we used `PARTITION BY` to partition by `flight_num` and `id`, we would see massive slowdown for any queries not on those columns, and writes would be prohibitively slow for this volume of data
- Now queries on flight number are improved
- Queries are a little slower on id now, however and we can look at the DAG to see why.

Note that, because  we had to read more files to satisfy this request. There is a (small) cost to clustering on more columns, so choose wisely.

In [0]:
%sql
SELECT AVG(ArrDelay) FROM dbacademy.flights_cluster_id_flightnum WHERE UniqueCarrier = 'TW'

In [0]:
%sql
SELECT AVG(ArrDelay) FROM dbacademy.flights_cluster_id_flightnum WHERE FlightNum = 1890

In [0]:
%sql
SELECT * FROM dbacademy.flights_cluster_id_flightnum WHERE id = 1125281431554


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>